# 1) Overview

This notebook now focuses on the corrected score-forward strategy search and the final winning configuration.

## Baseline
- Paper-style intraday Noise Area breakout on semi-hourly decision times.
- Gap-adjusted `UB/LB` bands from a 14-day minute-of-day sigma profile.
- Intraday stop uses the current band and VWAP.
- Flat by the close, with transaction costs included.

## Winning extension
- Keep the baseline signal and execution cadence at 30 minutes.
- Change the ML target horizon from 30 minutes to 60 minutes.
- Use a wider soft-sizing map: `size_floor=0.15`, `size_cap=2.25`.
- This was the only bounded search result that cleared Sharpe `1.2` on the corrected score-forward path.

## Benchmark retained for comparison
- `soft_size + market_vol_overlay` remains useful as a benchmark, but it no longer beats the new best `soft` configuration.


This notebook now also contains the detailed before-vs-after upgrade report that used to live separately.

# 2) Paper Context

The paper-inspired overlays reviewed in this project were:

1. **Moreira & Muir (Volatility-Managed Portfolios)**
   - idea: reduce or raise exposure based on broad market volatility;
   - implementation here: `market_vol_overlay` on top of the ML soft-sizing path.

2. **Barroso & Santa-Clara (Momentum Has Its Moments)**
   - idea: manage exposure using the strategy's own recent realized volatility;
   - implementation here: `strategy_vol_overlay` on top of the ML soft-sizing path.

3. **Daniel & Moskowitz (Momentum Crashes)**
   - idea: de-risk in panic / crash-like states;
   - implementation was tested historically, but it is no longer kept as a serious active method.

4. **Moskowitz, Ooi, Pedersen (Time-Series Momentum)**
   - idea: condition exposure on the larger trend state;
   - implementation was tested historically, but it is no longer kept as a serious active method.

A fifth tactical idea, **Zarattini & Pagani Fast Alpha**, was also prototyped as a lightweight entry-side overlay. It improved risk shape, but it did not beat baseline on the corrected source-level score-forward test.


# 3) Setup

In [ ]:
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = Path('/Users/ulianahusak/WUTIS_2026/intraday_momentum_ml')
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.config import load_config
from src.indicators import (
    compute_gap_adjusted_bands,
    compute_intraday_move_from_open,
    compute_sigma_profile,
    compute_vwap,
)
from src.preprocess import preprocess_bars

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

config = load_config()
DATA_DIR = config.data_dir if Path(config.data_dir).is_absolute() else REPO_ROOT / config.data_dir
REPORT_DIR = config.reports_dir / 'model_selection' if Path(config.reports_dir).is_absolute() else REPO_ROOT / config.reports_dir / 'model_selection'
PLOT_DIR = REPORT_DIR / 'plots'
TMP_DIR = REPORT_DIR / 'strategy_upgrade'
VENV_PYTHON = REPO_ROOT / '.venv/bin/python'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COLORS = {
    'baseline': '#4c4c4c',
    'soft': '#1f77b4',
    'mm': '#ff7f0e',
    'soft_before': '#9ecae1',
    'soft_now': '#1f77b4',
    'mm_before': '#fdd0a2',
    'mm_now': '#ff7f0e',
}

FORCE_RECOMPUTE_SERIOUS = False


# 4) Load Enriched Bars

The notebook uses `bars_enriched.parquet` when it already exists. If it does not, it rebuilds the enriched bars from `bars_raw.parquet` using the current preprocessing and indicator code.

In [ ]:
bars_path = DATA_DIR / 'bars_enriched.parquet'
raw_path = DATA_DIR / 'bars_raw.parquet'

if bars_path.exists():
    bars = pd.read_parquet(bars_path).copy()
else:
    raw = pd.read_parquet(raw_path).copy()
    clean = preprocess_bars(raw)
    bars = compute_intraday_move_from_open(clean)
    bars = compute_sigma_profile(bars, lookback_days=14)
    bars = compute_gap_adjusted_bands(bars)
    bars = compute_vwap(bars)
    bars.to_parquet(bars_path, index=False)

bars['date'] = pd.to_datetime(bars['date'])
print('Rows:', len(bars))
print('Date range:', bars['date'].min().date(), '->', bars['date'].max().date())
display(bars.head())


# 5) Historical Context: Retired Experiments

This table is loaded from the archived score-forward and extension report outputs. These methods are shown for research context only. They are **not** treated as active methods anymore because they either failed to beat baseline on corrected evaluation or were too brittle to keep.

In [ ]:
historical_path = REPORT_DIR / 'full_extension_summary.csv'
if historical_path.exists():
    historical = pd.read_csv(historical_path)
    retired = historical[historical['method'].isin([
        'bsc_strategy_vol_on_ml_source',
        'dm_panic_derisk_on_ml_source',
        'mop_trend_state_on_ml_source',
    ])].copy()
    retired = retired.sort_values(['sharpe', 'final_equity'], ascending=[False, False]).reset_index(drop=True)
    display(retired)
else:
    print('Archived historical summary not found:', historical_path)


# 6) Corrected Score-Forward Evaluation

The active comparison is now limited to:
- `baseline`
- `soft` = 60-minute ML label horizon + `size_floor=0.15`, `size_cap=2.25`
- `mm` = the same soft base plus market-vol overlay

This is the corrected protocol that should be trusted for selection.


In [ ]:
serious_summary_path = DATA_DIR / 'ml_scoreforward_summary.csv'
serious_splits_path = DATA_DIR / 'ml_scoreforward_splits.csv'
serious_curve_dir = DATA_DIR / 'ml_scoreforward'

if FORCE_RECOMPUTE_SERIOUS or not (serious_summary_path.exists() and serious_splits_path.exists()):
    subprocess.run([
        str(VENV_PYTHON),
        '-m',
        'src.cli',
        'backtest_ml_scoreforward',
        '--methods',
        'soft,mm',
    ], cwd=REPO_ROOT, check=True)

summary = pd.read_csv(serious_summary_path)
split_metrics = pd.read_csv(serious_splits_path)
serious_curves = {
    'baseline': pd.read_parquet(serious_curve_dir / 'baseline_equity_curve.parquet'),
    'soft': pd.read_parquet(serious_curve_dir / 'soft_equity_curve.parquet'),
    'mm': pd.read_parquet(serious_curve_dir / 'mm_equity_curve.parquet'),
}

summary_path = REPORT_DIR / 'notebook2_serious_summary.csv'
splits_path = REPORT_DIR / 'notebook2_serious_splits.csv'
summary.to_csv(summary_path, index=False)
split_metrics.to_csv(splits_path, index=False)

print('Saved summary to:', summary_path)
print('Saved split metrics to:', splits_path)
display(summary)


# 7) Why The New `soft` Strategy Improved

The new winner improved for two concrete reasons:

1. **The ML horizon was extended from 30 to 60 minutes**
   - the baseline still trades every 30 minutes;
   - the model now learns a less noisy forward outcome than the old 30-minute label.

2. **The soft-sizing map was widened materially**
   - weak candidates are sized down harder;
   - strong candidates are allowed to carry much more risk;
   - the signal quality was strong enough at the 60-minute horizon to support this wider map.

This is still a baseline-first strategy: direction comes from the baseline, and ML only controls sizing.


In [ ]:
baseline_row = summary.loc[summary['method'] == 'baseline'].iloc[0]
serious = summary.copy()
for metric in ['final_equity', 'sharpe', 'max_drawdown', 'turnover', 'total_costs']:
    serious[f'delta_{metric}_vs_baseline'] = serious[metric] - float(baseline_row[metric])
serious = serious.sort_values(['sharpe', 'final_equity'], ascending=[False, False]).reset_index(drop=True)
display(serious)


# 8) Plots

In [ ]:
plot_methods = ['baseline', 'soft', 'mm']
labels = {
    'baseline': 'baseline',
    'soft': 'soft_size (h60, 0.15/2.25)',
    'mm': 'soft_size + market_vol_overlay',
}

fig, ax = plt.subplots(figsize=(12, 5))
for method in plot_methods:
    eq = serious_curves[method].copy()
    ax.plot(pd.to_datetime(eq['date']), eq['equity'], label=labels[method], color=MODEL_COLORS[method])
ax.set_title('Corrected Score-Forward Equity Curves')
ax.set_xlabel('Date')
ax.set_ylabel('Equity')
ax.legend()
fig.tight_layout()
eq_plot_path = PLOT_DIR / 'notebook2_serious_equity_curves.png'
fig.savefig(eq_plot_path, dpi=150, bbox_inches='tight')
print('Saved equity plot to:', eq_plot_path)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary['method'], summary['sharpe'], color=[MODEL_COLORS[m] for m in summary['method']])
ax.set_title('Corrected Score-Forward Sharpe')
ax.set_xlabel('Method')
ax.set_ylabel('Sharpe')
fig.tight_layout()
bar_plot_path = PLOT_DIR / 'notebook2_serious_sharpe.png'
fig.savefig(bar_plot_path, dpi=150, bbox_inches='tight')
print('Saved Sharpe plot to:', bar_plot_path)


# 9) Final Takeaway

## Active methods kept in code
- `baseline`
- `ML soft_size` with a 60-minute label horizon and `size_floor=0.15`, `size_cap=2.25`
- `ML soft_size + market_vol_overlay` as a benchmark extension

## What the corrected evaluation now says
- the new `soft` configuration is the strongest validated strategy in the current codebase;
- it cleared Sharpe `1.2` on corrected score-forward evaluation;
- `market_vol_overlay` still beats baseline, but no longer beats the new `soft`.

## Practical interpretation
- the improvement came from changing the ML label horizon and widening the sizing response;
- the baseline still controls direction;
- the usable edge is still exposure management, not trade gating.


# 10) Strategy Upgrade Report

This section compares the previous serious setup against the current winning setup in one place.

It compares:
- the unchanged baseline;
- `soft_before` and `mm_before`: 30-minute ML horizon with the older milder sizing map;
- `soft_now` and `mm_now`: the current 60-minute ML horizon with the upgraded sizing map.

# 11) Old vs New Configuration

In [ ]:
config_table = pd.DataFrame([
    {'version': 'before', 'decision_freq_mins': 30, 'horizon_mins': 30, 'soft_floor': 0.50, 'soft_cap': 1.50, 'mm_floor': 0.50, 'mm_cap': 1.50},
    {'version': 'now', 'decision_freq_mins': 30, 'horizon_mins': 60, 'soft_floor': 0.15, 'soft_cap': 2.25, 'mm_floor': 0.75, 'mm_cap': 1.25},
])
display(config_table)


# 12) Load Cached Old/New Runs Or Recompute

In [ ]:
script = textwrap.dedent(f'''
from pathlib import Path
import sys
import pandas as pd

repo = Path({str(Path('/Users/ulianahusak/WUTIS_2026/intraday_momentum_ml'))!r})
sys.path.append(str(repo))

from src.scoreforward_eval import ScoreforwardConfig, run_ml_scoreforward_backtests

bars = pd.read_parquet(repo / 'data' / 'bars_enriched.parquet')
report_dir = repo / 'reports' / 'model_selection'
tmp_dir = report_dir / 'strategy_upgrade'
tmp_dir.mkdir(parents=True, exist_ok=True)

old_overrides = {{
    'soft': {{'size_floor': 0.50, 'size_cap': 1.50, 'regime_overlay': False}},
    'mm': {{
        'size_floor': 0.50,
        'size_cap': 1.50,
        'market_vol_overlay': True,
        'market_vol_lookback_days': 20,
        'market_vol_floor': 0.50,
        'market_vol_cap': 1.50,
        'regime_overlay': False,
    }},
}}

new_overrides = {{
    'soft': {{'size_floor': 0.15, 'size_cap': 2.25, 'regime_overlay': False}},
    'mm': {{
        'size_floor': 0.15,
        'size_cap': 2.25,
        'market_vol_overlay': True,
        'market_vol_lookback_days': 20,
        'market_vol_floor': 0.75,
        'market_vol_cap': 1.25,
        'regime_overlay': False,
    }},
}}

old_out = run_ml_scoreforward_backtests(
    bars,
    config=ScoreforwardConfig(decision_freq_mins=30, horizon_mins=30),
    methods=['soft', 'mm'],
    method_overrides=old_overrides,
)
new_out = run_ml_scoreforward_backtests(
    bars,
    config=ScoreforwardConfig(decision_freq_mins=30, horizon_mins=60),
    methods=['soft', 'mm'],
    method_overrides=new_overrides,
)

old_out['summary'].to_csv(tmp_dir / 'old_summary.csv', index=False)
new_out['summary'].to_csv(tmp_dir / 'new_summary.csv', index=False)

curve_parts = []
for label, out in [
    ('baseline', new_out['outputs']['baseline']['equity_curve']),
    ('soft_before', old_out['outputs']['soft']['equity_curve']),
    ('mm_before', old_out['outputs']['mm']['equity_curve']),
    ('soft_now', new_out['outputs']['soft']['equity_curve']),
    ('mm_now', new_out['outputs']['mm']['equity_curve']),
]:
    eq = out.copy()
    eq['label'] = label
    curve_parts.append(eq)
combined_curves = pd.concat(curve_parts, ignore_index=True)
combined_curves.to_parquet(tmp_dir / 'equity_curves.parquet', index=False)
print('Saved old/new summaries and curves to', tmp_dir)
''')

subprocess.run([str(VENV_PYTHON), '-c', script], check=True)

old_summary = pd.read_csv(TMP_DIR / 'old_summary.csv')
new_summary = pd.read_csv(TMP_DIR / 'new_summary.csv')
curves = pd.read_parquet(TMP_DIR / 'equity_curves.parquet')

display(old_summary)
display(new_summary)


# 13) Comparison Tables

In [ ]:
def enrich_summary(summary: pd.DataFrame, *, version: str, horizon_mins: int, soft_floor: float, soft_cap: float, mm_floor: float, mm_cap: float) -> pd.DataFrame:
    out = summary.copy()
    out['version'] = version
    out['decision_freq_mins'] = 30
    out['horizon_mins'] = horizon_mins
    out['size_floor'] = out['method'].map({'soft': soft_floor, 'mm': soft_floor, 'baseline': 1.0})
    out['size_cap'] = out['method'].map({'soft': soft_cap, 'mm': soft_cap, 'baseline': 1.0})
    out['mm_floor'] = out['method'].map({'mm': mm_floor}).fillna(0.0)
    out['mm_cap'] = out['method'].map({'mm': mm_cap}).fillna(0.0)
    label_map = {
        ('before', 'baseline'): 'baseline',
        ('before', 'soft'): 'soft_before',
        ('before', 'mm'): 'mm_before',
        ('now', 'baseline'): 'baseline',
        ('now', 'soft'): 'soft_now',
        ('now', 'mm'): 'mm_now',
    }
    out['label'] = [label_map[(version, method)] for method in out['method']]
    return out

before = enrich_summary(old_summary, version='before', horizon_mins=30, soft_floor=0.50, soft_cap=1.50, mm_floor=0.50, mm_cap=1.50)
now = enrich_summary(new_summary, version='now', horizon_mins=60, soft_floor=0.15, soft_cap=2.25, mm_floor=0.75, mm_cap=1.25)
comparison = pd.concat([before, now], ignore_index=True)
comparison = comparison.drop_duplicates(subset=['label'], keep='last').reset_index(drop=True)
comparison = comparison[['label', 'method', 'version', 'decision_freq_mins', 'horizon_mins', 'size_floor', 'size_cap', 'mm_floor', 'mm_cap', 'final_equity', 'sharpe', 'cagr_ish', 'max_drawdown', 'n_days', 'trades_count', 'turnover', 'total_costs']]

comparison_path = REPORT_DIR / 'strategy_upgrade_comparison.csv'
comparison.to_csv(comparison_path, index=False)
print('Saved comparison to:', comparison_path)
display(comparison)


In [ ]:
baseline_row = comparison.loc[comparison['label'] == 'baseline'].iloc[0]
soft_before_row = comparison.loc[comparison['label'] == 'soft_before'].iloc[0]

deltas = comparison.copy()
for metric in ['final_equity', 'sharpe', 'cagr_ish', 'max_drawdown', 'turnover', 'total_costs']:
    deltas[f'delta_{metric}_vs_baseline'] = deltas[metric] - float(baseline_row[metric])
    deltas[f'delta_{metric}_vs_soft_before'] = deltas[metric] - float(soft_before_row[metric])

delta_path = REPORT_DIR / 'strategy_upgrade_deltas.csv'
deltas.to_csv(delta_path, index=False)
print('Saved deltas to:', delta_path)
display(deltas[['label', 'final_equity', 'sharpe', 'max_drawdown', 'delta_final_equity_vs_baseline', 'delta_sharpe_vs_baseline', 'delta_final_equity_vs_soft_before', 'delta_sharpe_vs_soft_before']])

# 14) Before vs Now Plots

In [ ]:
labels = {
    'baseline': 'baseline',
    'soft_before': 'soft_before (h30, 0.50/1.50)',
    'mm_before': 'mm_before (h30, 0.50/1.50)',
    'soft_now': 'soft_now (h60, 0.15/2.25)',
    'mm_now': 'mm_now (h60, MM 0.75/1.25)',
}

fig, ax = plt.subplots(figsize=(12, 5))
for key in ['baseline', 'soft_before', 'mm_before', 'soft_now', 'mm_now']:
    eq = curves.loc[curves['label'] == key].copy()
    ax.plot(pd.to_datetime(eq['date']), eq['equity'], label=labels[key], color=MODEL_COLORS[key])
ax.set_title('Before vs Now: Score-Forward Equity Curves')
ax.set_xlabel('Date')
ax.set_ylabel('Equity')
ax.legend()
fig.tight_layout()
eq_path = PLOT_DIR / 'strategy_upgrade_equity_curves.png'
fig.savefig(eq_path, dpi=150, bbox_inches='tight')
print('Saved equity curves to:', eq_path)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for key in ['baseline', 'soft_before', 'mm_before', 'soft_now', 'mm_now']:
    eq = curves.loc[curves['label'] == key].copy()
    dd = eq['equity'] / eq['equity'].cummax() - 1.0
    ax.plot(pd.to_datetime(eq['date']), dd, label=labels[key], color=MODEL_COLORS[key])
ax.set_title('Before vs Now: Drawdown Curves')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.legend()
fig.tight_layout()
dd_path = PLOT_DIR / 'strategy_upgrade_drawdowns.png'
fig.savefig(dd_path, dpi=150, bbox_inches='tight')
print('Saved drawdown curves to:', dd_path)


In [ ]:
metric_order = ['baseline', 'soft_before', 'mm_before', 'soft_now', 'mm_now']
metric_df = comparison.set_index('label').loc[metric_order].reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(metric_df['label'], metric_df['sharpe'], color=[MODEL_COLORS[k] for k in metric_df['label']])
axes[0].set_title('Sharpe')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(metric_df['label'], metric_df['final_equity'], color=[MODEL_COLORS[k] for k in metric_df['label']])
axes[1].set_title('Final Equity')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(metric_df['label'], metric_df['max_drawdown'], color=[MODEL_COLORS[k] for k in metric_df['label']])
axes[2].set_title('Max Drawdown')
axes[2].tick_params(axis='x', rotation=30)

fig.tight_layout()
bar_path = PLOT_DIR / 'strategy_upgrade_metric_bars.png'
fig.savefig(bar_path, dpi=150, bbox_inches='tight')
print('Saved metric bars to:', bar_path)


# 15) Update Interpretation

## What stayed the same
- The baseline still controls direction.
- Decision cadence stayed at 30 minutes.
- The trade count stayed unchanged, so the improvement did not come from taking more signals.

## What changed
1. The ML label horizon was extended from 30 to 60 minutes.
2. The soft-sizing map widened from `0.50..1.50` to `0.15..2.25`.
3. The market-vol overlay remained a benchmark, but the upgraded soft strategy became the stronger method.

## Practical conclusion
- The Sharpe improvement came from sizing the same baseline trades more selectively and more aggressively.
- The tradeoff is a larger drawdown than the earlier, milder soft-sizing setup.